# Docling Structure Inspection Notebook

Docling extracts document structure elements such as text blocks, tables, and pictures. This notebook inspects one PolyDocBench scan, compares Docling output with transformed GT, and visualizes matched and unmatched layout elements.

Workflow:

1. Select one noisy scan generated by the experiment preparation pipeline.
2. Build visible PolyDocBench GT structure elements.
3. Run Docling or reuse a saved Docling JSON output.
4. Compute structure detection, geometry, and type metrics.
5. Draw GT and Docling overlays for manual inspection.


## 1. Environment Setup

Install optional dependencies before starting Jupyter:

```powershell
uv pip install -e ".[structure,noise,dev]"
```

Docling can be slow on the first image because it initializes local model pipelines. Set `REUSE_EXISTING_PREDICTION = True` after a successful run.


In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

from IPython.display import Image as IPyImage, Markdown, display
from PIL import Image, ImageDraw

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")


## 2. Select One Scan

The input scan is taken from `outputs/experiments/tesseract_quality` because that pipeline already produced noisy images and transformed GT. Tesseract predictions are not used for structure evaluation.


In [ ]:
LANGUAGE_CODE = "en"
ARTICLE_ID = "history_russia"
TEMPLATE = "scientific_paper"
PAGE_NUMBER = 1
NOISE_PROFILE = "medium_scan"
VARIANT = 0

INPUT_ROOT = PROJECT_ROOT / "outputs" / "experiments" / "tesseract_quality"
RESULT_ROOT = PROJECT_ROOT / "outputs" / "experiments" / "docling_structure"
PAGE_DIR = f"page_{PAGE_NUMBER:03d}"
STEM = f"{NOISE_PROFILE}_{VARIANT}"
SOURCE_CASE = INPUT_ROOT / LANGUAGE_CODE / ARTICLE_ID / TEMPLATE / "noisy" / PAGE_DIR
RESULT_CASE = RESULT_ROOT / LANGUAGE_CODE / ARTICLE_ID / TEMPLATE / "noisy" / PAGE_DIR
SCAN_PATH = SOURCE_CASE / f"{STEM}.jpg"
GT_PATH = SOURCE_CASE / f"{STEM}_gt.json"
RAW_OUTPUT_PATH = RESULT_CASE / f"{STEM}_docling_raw.json"
PREDICTION_PATH = RESULT_CASE / f"{STEM}_docling_structure.json"
MATCHES_PATH = RESULT_CASE / f"{STEM}_structure_matches.json"
OVERLAY_PATH = RESULT_CASE / f"{STEM}_structure_overlay.jpg"

assert SCAN_PATH.exists(), f"Missing scan: {SCAN_PATH}"
assert GT_PATH.exists(), f"Missing GT: {GT_PATH}"
print("Scan:", SCAN_PATH)
print("GT:", GT_PATH)
print("Prediction:", PREDICTION_PATH)


## 3. Build Visible GT Structure

The GT adapter groups visible text lines by source block and keeps non-text elements such as images, tables, and formulas. All boxes are evaluated in transformed image coordinates.


In [ ]:
from polydocbench.eval import extract_gt_structure_elements, load_gt

gt = load_gt(GT_PATH)
gt_elements = extract_gt_structure_elements(gt, page_number=PAGE_NUMBER)

print("GT structure elements:", len(gt_elements))
for index, element in enumerate(gt_elements[:20], start=1):
    print(f"{index:02d} | {element['type']} | {element['id']} | {element.get('text', '')[:90]}")

display(IPyImage(filename=str(SCAN_PATH)))


## 4. Run Or Reuse Docling

Docling raw output is saved separately from the normalized PolyDocBench structure prediction. This makes long runs reusable and auditable.


In [ ]:
from polydocbench.eval import extract_docling_structure, parse_docling_structure

REUSE_EXISTING_PREDICTION = True

RESULT_CASE.mkdir(parents=True, exist_ok=True)
if REUSE_EXISTING_PREDICTION and PREDICTION_PATH.exists():
    predicted_elements = json.loads(PREDICTION_PATH.read_text(encoding="utf-8"))
elif REUSE_EXISTING_PREDICTION and RAW_OUTPUT_PATH.exists():
    predicted_elements = parse_docling_structure(
        json.loads(RAW_OUTPUT_PATH.read_text(encoding="utf-8")), page_number=PAGE_NUMBER
    )
    PREDICTION_PATH.write_text(json.dumps(predicted_elements, ensure_ascii=False, indent=2), encoding="utf-8")
else:
    predicted_elements = extract_docling_structure(
        SCAN_PATH,
        raw_output_path=RAW_OUTPUT_PATH,
        page_number=PAGE_NUMBER,
    )
    PREDICTION_PATH.write_text(json.dumps(predicted_elements, ensure_ascii=False, indent=2), encoding="utf-8")

print("Docling structure elements:", len(predicted_elements))
for index, element in enumerate(predicted_elements[:20], start=1):
    print(f"{index:02d} | {element['type']} | {element['id']} | {element.get('text', '')[:90]}")


## 5. Evaluate Structure

The primary metrics are detection F1, mean IoU, type accuracy, and a weighted `structure_score`.


In [ ]:
from polydocbench.eval import evaluate_structure, structure_matches_to_dicts

IOU_THRESHOLD = 0.50
metrics, matches = evaluate_structure(gt_elements, predicted_elements, iou_threshold=IOU_THRESHOLD)
MATCHES_PATH.write_text(json.dumps(structure_matches_to_dicts(matches), ensure_ascii=False, indent=2), encoding="utf-8")

display(Markdown("### Structure Metrics"))
display(metrics)

print("\n--- First matches ---")
for index, match in enumerate(matches[:20], start=1):
    pred_id = match.prediction.get("id") if match.prediction else "<NO MATCH>"
    pred_type = match.prediction.get("type") if match.prediction else "<NO MATCH>"
    print(
        f"{index:02d} | GT={match.gt['id']}:{match.gt['type']} | "
        f"Pred={pred_id}:{pred_type} | IoU={match.iou:.3f} | type_correct={match.type_correct}"
    )


## 6. Draw Overlay

Red rectangles show GT structure elements. Green rectangles show Docling predictions. The overlay is diagnostic; metrics are computed from the normalized element lists above.


In [ ]:
def draw_structure_overlay(image_path: Path, gt_elements: list[dict], predicted_elements: list[dict], output_path: Path) -> Path:
    image = Image.open(image_path).convert("RGB")
    draw = ImageDraw.Draw(image)

    for element in gt_elements:
        bbox = element.get("bbox")
        if not bbox:
            continue
        x0, y0 = bbox["x"], bbox["y"]
        x1, y1 = x0 + bbox["width"], y0 + bbox["height"]
        draw.rectangle([x0, y0, x1, y1], outline="red", width=3)

    for element in predicted_elements:
        bbox = element.get("bbox")
        if not bbox:
            continue
        x0, y0 = bbox["x"], bbox["y"]
        x1, y1 = x0 + bbox["width"], y0 + bbox["height"]
        draw.rectangle([x0, y0, x1, y1], outline="lime", width=2)

    output_path.parent.mkdir(parents=True, exist_ok=True)
    image.save(output_path, quality=95)
    return output_path

overlay = draw_structure_overlay(SCAN_PATH, gt_elements, predicted_elements, OVERLAY_PATH)
print("Overlay:", overlay)
display(IPyImage(filename=str(overlay)))


## 7. Batch Runner Equivalent

Use the script below for the same workflow over many generated scans:

```powershell
uv run python scripts\run_docling_structure_experiment.py `
  --input-dir outputs\experiments\tesseract_quality `
  --output-dir outputs\experiments\docling_structure `
  --languages en `
  --article-ids history_russia `
  --templates scientific_paper `
  --profiles medium_scan `
  --page-numbers 1 `
  --max-images 1 `
  --reuse
```
